### **BILBIOTECAS**

In [24]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
import pandas as pd
import numpy as np 

from feature_engine.selection import DropConstantFeatures, DropDuplicateFeatures, SmartCorrelatedSelection

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

from projeto_ml.pipeline import Pipeline_Data

pd.set_option('display.max_columns', None)

In [35]:
LOCAL_DADOS = '../data/gold'

In [36]:
df = pd.read_parquet(f'{LOCAL_DADOS}/churn.parquet')

df.head(2)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,0


In [27]:
[print(i) for i in df.columns if 'TotalCharges' in i];

In [28]:
df_feat = df.copy()   

# 1. Tempo e Valor
df_feat['Is_New_Customer'] = (df_feat['tenure'] <= 6).astype(int)    

# 2. Engajamento
servicos = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df_feat['Total_Services'] = df_feat[servicos].apply(lambda x: (x == 'Yes').sum(), axis=1)
df_feat['Has_Internet'] = (df_feat['InternetService'] != 'No').astype(int)    

# 3. Contrato e Pagamento
df_feat['Is_Monthly_Contract'] = (df_feat['Contract'] == 'Month-to-month').astype(int)
df_feat['Is_Auto_Payment'] = df_feat['PaymentMethod'].isin(['Bank transfer (automatic)', 'Credit card (automatic)']).astype(int)    

# 4. Perfil Familiar
df_feat['Family_Size'] = 1 + (df_feat['Partner'] == 'Yes').astype(int) + (df_feat['Dependents'] == 'Yes').astype(int)    

# 5. Risco Financeiro
df_feat['Charge_to_Tenure_Ratio'] = df_feat['MonthlyCharges'] / (df_feat['tenure'] + 1)    


### **Pipeline Dados**

In [29]:
X = df_feat.drop(columns=['Churn'])
y = df_feat['Churn']

pipe = Pipeline_Data(X=X, y=y, test_size=.30)

X_train, y_train, X_val, y_val, X_test, y_test, pipeline = pipe.split_data()

Shape original do treino: (4930, 26)
Shape após limpeza: (4930, 26)

Shape test: (1057, 26)

Shape val: (1056, 26)

--- Features Constantes Removidas ---
[]

--- Features Correlacionadas Removidas ---
[]

--- Pares de Features Correlacionadas (Correlação > 0.85) ---
[]

--- Features Duplicadas Removidas ---
set()

--- Grupos de Features Duplicadas ---
[]


#### **BASE_TREINAMENTO**

In [ ]:
X_train['Churn'] = y_train
X_train.to_parquet(f'{LOCAL_DADOS}/train.parquet')

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Is_New_Customer,Total_Services,Has_Internet,Is_Monthly_Contract,Is_Auto_Payment,Family_Size,Charge_to_Tenure_Ratio,Churn
5557,6918-UMQCG,Female,0,No,No,5,Yes,No,Fiber optic,No,No,No,No,Yes,No,Month-to-month,No,Electronic check,80.20,1,2,1,1,0,1,13.366667,0
2270,4140-MUHUG,Female,1,No,No,3,Yes,No,Fiber optic,No,No,Yes,No,Yes,No,Month-to-month,Yes,Electronic check,86.85,1,3,1,1,0,1,21.712500,1
6930,5570-PTWEH,Female,0,Yes,No,3,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Credit card (automatic),75.15,1,2,1,1,1,2,18.787500,1
2257,2054-PJOCK,Female,0,No,No,60,Yes,Yes,DSL,No,No,Yes,Yes,Yes,Yes,One year,No,Credit card (automatic),80.55,0,6,1,0,1,1,1.320492,0
898,4315-MURBD,Female,0,No,No,12,Yes,No,Fiber optic,Yes,No,No,Yes,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),98.90,0,5,1,1,1,1,7.607692,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4250,0565-JUPYD,Male,0,No,No,63,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,No,One year,No,Credit card (automatic),104.50,0,7,1,0,1,1,1.632812,0
1488,8695-WDYEA,Male,0,No,No,1,Yes,No,DSL,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,51.25,1,2,1,1,0,1,25.625000,0
6303,6308-CQRBU,Female,0,Yes,No,71,Yes,Yes,Fiber optic,No,Yes,Yes,Yes,Yes,Yes,Two year,No,Electronic check,109.25,0,7,1,0,0,2,1.517361,0
2710,3594-BDSOA,Female,0,Yes,No,24,Yes,No,No,No,No,No,No,No,No,One year,No,Credit card (automatic),20.40,0,1,0,0,1,2,0.816000,0


#### **BASE_VALIDACAO**

In [ ]:
X_test['Churn'] = y_test
X_test.to_parquet(f'{LOCAL_DADOS}/test.parquet')

#### **BASE_TESTE***

In [ ]:
X_val['Churn'] = y_val
X_val.to_parquet(f'{LOCAL_DADOS}/val.parquet')